# EfficientNet-B0 v3 — 채널 확장 + CutMix + Mixup
cuda:3 사용 / 셀 순서대로 실행

In [1]:
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(3)}')

Device: cuda:3
GPU: NVIDIA RTX 2000 Ada Generation


In [2]:
CFG = dict(
    data_root    = './student_data',
    batch_size   = 256,
    num_epochs   = 200,
    lr           = 0.1,
    momentum     = 0.9,
    weight_decay = 1e-4,
    label_smooth = 0.1,
    dropout      = 0.2,
    num_workers  = 4,
    cutmix_alpha = 1.0,
    mixup_alpha  = 0.2,
    output_pt    = 'my_model_v3.pt',
)
print('CFG OK')

CFG OK


In [9]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.SiLU(),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), -1, 1, 1)


class MBConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, stride, expand_ratio, se_ratio=0.25):
        super().__init__()
        self.use_skip = (stride == 1 and in_ch == out_ch)
        mid_ch = in_ch * expand_ratio
        pad = (kernel - 1) // 2
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, mid_ch, 1, bias=False),
                       nn.BatchNorm2d(mid_ch), nn.SiLU()]
        layers += [nn.Conv2d(mid_ch, mid_ch, kernel, stride, pad, groups=mid_ch, bias=False),
                   nn.BatchNorm2d(mid_ch), nn.SiLU()]
        layers.append(SEBlock(mid_ch, reduction=max(1, int(1 / se_ratio))))
        layers += [nn.Conv2d(mid_ch, out_ch, 1, bias=False),
                   nn.BatchNorm2d(out_ch)]
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        out = self.block(x)
        return x + out if self.use_skip else out


class EfficientNetB0(nn.Module):
    # v3: 채널 확장 버전 (v2 대비 capacity 향상)
    STAGES = [
        (1, 16,  1, 3, 1),
        (6, 24,  2, 3, 2),
        (6, 40,  2, 5, 2),
        (6, 80,  3, 3, 1),
        (6, 96,  3, 5, 1),   # 104 → 96
        (6, 144, 4, 5, 2),   # 160 → 144
        (6, 240, 1, 3, 1),   # 256 → 240
    ]
    def __init__(self, num_classes=200, dropout=0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.SiLU(),
        )
        blocks, in_ch = [], 32
        for expand, out_ch, n_layers, k, s in self.STAGES:
            for i in range(n_layers):
                blocks.append(MBConv(in_ch, out_ch, k, s if i == 0 else 1, expand))
                in_ch = out_ch
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.Conv2d(240, 960, 1, bias=False),
            nn.BatchNorm2d(960), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(960, num_classes),
        )
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))


model = EfficientNetB0(num_classes=200, dropout=CFG['dropout'])
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,}')
assert params <= 5_000_000, f'파라미터 초과: {params:,}'
print('파라미터 OK')

Parameters: 4,773,032
파라미터 OK


In [10]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, classes=200, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.cls = classes
    def forward(self, pred, target):
        confidence = 1.0 - self.smoothing
        smooth_val = self.smoothing / (self.cls - 1)
        log_probs = F.log_softmax(pred, dim=-1)
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, smooth_val)
            true_dist.scatter_(1, target.unsqueeze(1), confidence)
        return -(true_dist * log_probs).sum(dim=-1).mean()
print('Loss OK')

Loss OK


In [11]:
class TinyImageNetVal(Dataset):
    def __init__(self, root, transform=None):
        self.img_dir   = os.path.join(root, 'images')
        self.transform = transform
        self.samples   = []
        with open(os.path.join(root, 'labels.txt')) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                fname, cls = line.split('\t')
                self.samples.append((fname, int(cls)))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        fname, label = self.samples[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label


def get_loaders(cfg):
    train_tf = transforms.Compose([
        transforms.RandomCrop(64, padding=8),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    train_ds = datasets.ImageFolder(
        os.path.join(cfg['data_root'], 'train'), transform=train_tf)
    val_ds = TinyImageNetVal(
        os.path.join(cfg['data_root'], 'public_val'), transform=val_tf)
    print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')
    pw = cfg['num_workers'] > 0
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                              num_workers=cfg['num_workers'], pin_memory=True, persistent_workers=pw)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=cfg['num_workers'], pin_memory=True, persistent_workers=pw)
    return train_loader, val_loader

print('Dataset OK')

Dataset OK


In [12]:
def cutmix(imgs, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    _, _, H, W = imgs.shape
    cx, cy = np.random.randint(W), np.random.randint(H)
    w = int(W * math.sqrt(1 - lam))
    h = int(H * math.sqrt(1 - lam))
    x1, x2 = max(cx - w//2, 0), min(cx + w//2, W)
    y1, y2 = max(cy - h//2, 0), min(cy + h//2, H)
    imgs = imgs.clone()
    imgs[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return imgs, labels, labels[idx], lam

def mixup(imgs, labels, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    imgs = lam * imgs + (1 - lam) * imgs[idx]
    return imgs, labels, labels[idx], lam


def train_one_epoch(model, loader, optimizer, criterion, scaler, cfg):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        r = np.random.rand()
        if r < 0.4:
            imgs, la, lb, lam = cutmix(imgs, labels, cfg['cutmix_alpha'])
        elif r < 0.7:
            imgs, la, lb, lam = mixup(imgs, labels, cfg['mixup_alpha'])
        else:
            la, lb, lam = labels, labels, 1.0
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss = lam * criterion(logits, la) + (1 - lam) * criterion(logits, lb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == la).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        correct += (model(imgs).argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return correct / total

print('Train/Eval 함수 OK')

Train/Eval 함수 OK


In [ ]:
cfg = CFG
model = EfficientNetB0(num_classes=200, dropout=cfg['dropout']).to(DEVICE)

criterion = LabelSmoothingCE(200, cfg['label_smooth'])
optimizer = optim.SGD(model.parameters(), lr=cfg['lr'],
                      momentum=cfg['momentum'], weight_decay=cfg['weight_decay'], nesterov=True)

def lr_lambda(epoch):
    warmup = 10
    if epoch < warmup: return (epoch + 1) / warmup
    progress = (epoch - warmup) / (cfg['num_epochs'] - warmup)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler('cuda')
train_loader, val_loader = get_loaders(cfg)

best_acc = 0.0
for epoch in range(cfg['num_epochs']):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, cfg)
    val_acc = evaluate(model, val_loader)
    scheduler.step()
    lr_now = optimizer.param_groups[0]['lr']
    print(f'[{epoch+1:3d}/{cfg["num_epochs"]}] loss={tr_loss:.4f}  tr={tr_acc:.4f}  val={val_acc:.4f}  lr={lr_now:.5f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_weights_v3.pth')
        print(f'  ✓ best: {best_acc:.4f}')

print(f'\n학습 완료. Best val acc: {best_acc:.4f}')

Train: 100,000  Val: 5,000
[  1/200] loss=5.2627  tr=0.0086  val=0.0188  lr=0.02000
  ✓ best: 0.0188
[  2/200] loss=5.0465  tr=0.0247  val=0.0634  lr=0.03000
  ✓ best: 0.0634
[  3/200] loss=4.8168  tr=0.0490  val=0.0916  lr=0.04000
  ✓ best: 0.0916
[  4/200] loss=4.6307  tr=0.0728  val=0.1428  lr=0.05000
  ✓ best: 0.1428
[  5/200] loss=4.4795  tr=0.0949  val=0.1716  lr=0.06000
  ✓ best: 0.1716
[  6/200] loss=4.3971  tr=0.1116  val=0.1752  lr=0.07000
  ✓ best: 0.1752
[  7/200] loss=4.2996  tr=0.1249  val=0.1998  lr=0.08000
  ✓ best: 0.1998
[  8/200] loss=4.1725  tr=0.1476  val=0.2478  lr=0.09000
  ✓ best: 0.2478
[  9/200] loss=4.0321  tr=0.1732  val=0.2858  lr=0.10000
  ✓ best: 0.2858
[ 10/200] loss=4.0328  tr=0.1779  val=0.2878  lr=0.10000
  ✓ best: 0.2878
[ 11/200] loss=3.9447  tr=0.2004  val=0.3184  lr=0.09999
  ✓ best: 0.3184
[ 12/200] loss=3.8096  tr=0.2163  val=0.3458  lr=0.09997
  ✓ best: 0.3458
[ 13/200] loss=3.7550  tr=0.2333  val=0.3560  lr=0.09994
  ✓ best: 0.3560
[ 14/200] l

In [ ]:
model.load_state_dict(torch.load('best_weights_v3.pth', map_location='cpu'))
model.eval().cpu()
dummy  = torch.randn(1, 3, 64, 64)
traced = torch.jit.trace(model, dummy)
torch.jit.save(traced, cfg['output_pt'])

loaded  = torch.jit.load(cfg['output_pt'], map_location='cpu').eval()
p_count = sum(p.numel() for p in loaded.parameters())
with torch.no_grad():
    out = loaded(dummy)
assert out.shape == (1, 200)
assert p_count <= 5_000_000
print(f'✓ Export OK — {p_count:,} params, shape {tuple(out.shape)}')